# CS570 — Project Deliverable 4: Nonlinear Classification Model
**Team:** Pentanet  
**Members:** AZATBEK ISMAILOV, FSEHAYE MEDHANIE, NILA KO, KHAING MIN HTWE, YUEXUAN LU  
**Date:** April 22, 2026  
**Instructor:** Dr. Ragnar Lesch | SFBU CS570 — Big Data Processing & Analytics (Spring 2026)

---
**Goal:** Build a nonlinear Spark MLlib Pipeline using Gradient-Boosted Trees (GBTClassifier) to predict `high_rating` (Rating ≥ 4). Improve on the D3 Logistic Regression baseline (AUC-PR 0.8150, F1 0.7165) and generate holdout predictions.

### Notebook Structure

| Part | Description |
|---|---|
| Part 1 | Model Construction — Algorithm selection, Pipeline (no scaler), Hyperparameter tuning |
| Part 2 | Model Evaluation — 5 metrics, D3 comparison table, confusion matrix, feature importance |
| Part 3 | Holdout Predictions — Retrain on full data, produce predictions.csv (100,021 rows) |
| Part 4 | Reflection — 2 written responses |

**Training Data:** `data/raw/ratings_train.dat` (900,188 rows) · `users.dat` · `movies.dat`  
**Holdout:** `data/raw/holdout_test.csv` (100,021 rows — no Rating column, no target label)  
**Framework:** Apache Spark / PySpark | Python 3.x

---
## Configuration



In [1]:
import sys
print(sys.executable)

d:\SFBU\Spring_semester_2026\CS570\Project-CS-570\.venv\Scripts\python.exe


In [2]:
import os, warnings
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
plt.switch_backend('agg')

PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..', '..'))
DATA_DIR     = os.path.join(PROJECT_ROOT, 'data', 'raw')

# D4: ratings_train.dat  (NOT ratings.dat)
RATINGS_TRAIN_PATH = os.path.join(DATA_DIR, 'ratings_train.dat')
USERS_PATH         = os.path.join(DATA_DIR, 'users.dat')
MOVIES_PATH        = os.path.join(DATA_DIR, 'movies.dat')
HOLDOUT_PATH       = os.path.join(DATA_DIR, 'holdout_test.csv')
OUTPUT_PATH        = os.path.join(PROJECT_ROOT, 'notebooks', 'D4', 'predictions.csv')

for label, path in [
    ('ratings_train (D4 train)', RATINGS_TRAIN_PATH),
    ('users',                    USERS_PATH),
    ('movies',                   MOVIES_PATH),
    ('holdout_test',             HOLDOUT_PATH),
]:
    status = 'FOUND    ' if os.path.exists(path) else 'NOT FOUND'
    print(f'{status} [{label}]: {path}')

FOUND     [ratings_train (D4 train)]: d:\SFBU\Spring_semester_2026\CS570\Project-CS-570\data\raw\ratings_train.dat
FOUND     [users]: d:\SFBU\Spring_semester_2026\CS570\Project-CS-570\data\raw\users.dat
FOUND     [movies]: d:\SFBU\Spring_semester_2026\CS570\Project-CS-570\data\raw\movies.dat
FOUND     [holdout_test]: d:\SFBU\Spring_semester_2026\CS570\Project-CS-570\data\raw\holdout_test.csv


---
## SparkSession

In [3]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName('CS570-D4-Pentanet-GBT')
    .master('local[*]')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.driver.memory', '4g')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('ERROR')
print('Spark version:', spark.version)

Spark version: 3.5.0


---
## Part 1: Model Construction 

### Part 1a: Model Selection — Gradient-Boosted Trees 

**Algorithm chosen:** `GBTClassifier` (`pyspark.ml.classification.GBTClassifier`)

---

#### How GBT Differs from Logistic Regression

Logistic Regression (D3) models log-odds as a single **linear** combination of features:

```
log(p / 1-p) = w0 + w1*x1 + w2*x2 + ... + w12*x12
```

One global weight per feature. It **cannot** express:
- Nonlinear thresholds: `if movie_avg_rating > 4.2 AND user_avg_rating > 3.8 → nearly certain high rating`
- Conditional interactions: `is_action matters for young users (Age 18) but not older users (Age 56)`

GBT builds an **ensemble of decision trees sequentially**. Tree 1 makes predictions. Tree 2 focuses specifically on the rows Tree 1 got wrong. Tree 3 focuses on what Trees 1+2 still got wrong — and so on up to `maxIter` trees. Each tree corrects the **residual errors** (gradients of the loss) of all previous trees. The final prediction is the weighted sum of all tree outputs. This allows GBT to:

1. **Capture nonlinear decision boundaries** — any piecewise-constant region in feature space, not just a hyperplane.
2. **Discover feature interactions automatically** — a tree can split first on `movie_avg_rating` and then on `Age` within that sub-branch, modeling the conditional interaction without explicit feature engineering.
3. **Eliminate the suppressor variable problem** — in D3, `user_movie_interaction` received coefficient –0.896 because LR double-counted its components (`user_avg_rating` × `movie_avg_rating`). GBT splits on individual features at each node; collinearity does not cause sign reversal.

**No StandardScaler needed:** GBT splits by threshold comparisons (`feature > value`). The absolute scale of a feature does not affect which threshold is best — a split at `movie_avg_rating > 3.8` is equally valid on raw or scaled values. Scaling is only required for gradient-descent-based learners like LR.

---

#### Why GBT Should Outperform D3 on This Dataset — Three Concrete Reasons

1. **Rating behavior is nonlinear.** A user with `user_avg_rating=4.0` watching a movie with `movie_avg_rating=4.5` should nearly guarantee a high rating. A user at 3.5 watching a 3.5-avg movie is genuinely uncertain. LR cannot express *both high ⟹ almost certain* as a threshold region; GBT discovers it as a split sequence.

2. **Genre × Age interaction.** In D3 Part 4 we noted that Action films may rate higher for younger users (Age code 18–25) but lower for older users (50–56), while Drama reverses this pattern. LR assigns one weight to `is_action` and one to `Age` — it cannot learn their joint effect. GBT finds the conditional split: `if is_action=1 → split on Age`.

3. **D3's strongest predictor was a suppressor.** `user_movie_interaction` (D2 Pearson rank #1, |r|=0.484) carried a **negative** LR coefficient (–0.896) due to multicollinearity. GBT does not have this problem — it ranks each feature by its actual split contribution, giving an honest importance signal.

---
**D3 targets to beat:** AUC-PR = 0.8150 | Accuracy = 0.7215 | F1 = 0.7165

---
## Data Loading

In [4]:
from pyspark.sql.types import (
    StructType, StructField, IntegerType, LongType, StringType, FloatType
)

RATINGS_SCHEMA = StructType([
    StructField('UserID',    IntegerType(), nullable=False),
    StructField('MovieID',   IntegerType(), nullable=False),
    StructField('Rating',    FloatType(),   nullable=False),
    StructField('Timestamp', LongType(),    nullable=False),
])
USERS_SCHEMA = StructType([
    StructField('UserID',     IntegerType(), nullable=False),
    StructField('Gender',     StringType(),  nullable=False),
    StructField('Age',        IntegerType(), nullable=False),
    StructField('Occupation', IntegerType(), nullable=False),
    StructField('ZipCode',    StringType(),  nullable=True),
])
MOVIES_SCHEMA = StructType([
    StructField('MovieID', IntegerType(), nullable=False),
    StructField('Title',   StringType(),  nullable=False),
    StructField('Genres',  StringType(),  nullable=False),
])

# ratings_train.dat — expect ~900,188 rows (NOT 1,000,209)
ratings = spark.read.option('sep', '::').schema(RATINGS_SCHEMA).csv(RATINGS_TRAIN_PATH)
users   = spark.read.option('sep', '::').schema(USERS_SCHEMA).csv(USERS_PATH)
movies  = spark.read.option('sep', '::').schema(MOVIES_SCHEMA).csv(MOVIES_PATH)

print(f'Ratings (train) : {ratings.count():>10,}  <- expect ~900,188')
print(f'Users           : {users.count():>10,}')
print(f'Movies          : {movies.count():>10,}')

Ratings (train) :    900,188  <- expect ~900,188
Users           :      6,040
Movies          :      3,883


### Join Tables

In [5]:
from pyspark.sql import functions as F

joined = (
    ratings
    .join(users,  on='UserID',  how='inner')
    .join(movies, on='MovieID', how='inner')
).cache()

print(f'Joined rows : {joined.count():,}')
print(f'Columns     : {len(joined.columns)}')
joined.show(3)

Joined rows : 900,188
Columns     : 10
+-------+------+------+---------+------+---+----------+-------+-------------------+------------+
|MovieID|UserID|Rating|Timestamp|Gender|Age|Occupation|ZipCode|              Title|      Genres|
+-------+------+------+---------+------+---+----------+-------+-------------------+------------+
|     69|   442|   4.0|997228510|     M| 25|         1|  55105|      Friday (1995)|      Comedy|
|   2374|  2976|   3.0|971022300|     M| 18|        20|  86001|     Gung Ho (1986)|Comedy|Drama|
|   3911|  2748|   5.0|973208387|     M| 25|         4|  85719|Best in Show (2000)|      Comedy|
+-------+------+------+---------+------+---+----------+-------+-------------------+------------+
only showing top 3 rows



---
## Feature Engineering

Same 12-feature set as D3. Two aggregate tables (`movie_stats`, `user_stats`) are computed here from training data and **reused for the holdout in Part 3** — recomputing from holdout would be leakage (and impossible: holdout has no Rating column).

In [6]:
df = joined

# Target: Rating >= 4 -> 1 (high rating), else 0
df = df.withColumn('high_rating', F.when(F.col('Rating') >= 4, 1).otherwise(0))

# Movie-level aggregates -- cached separately so Part 3 can join them to holdout
movie_stats = df.groupBy('MovieID').agg(
    F.avg('Rating').alias('movie_avg_rating'),
    F.count('Rating').alias('movie_popularity'),
).cache()

df = df.join(movie_stats, on='MovieID', how='left')
df = df.withColumn('log_movie_popularity', F.log(F.col('movie_popularity') + 1))
df = df.withColumn(
    'release_year',
    F.regexp_extract(F.col('Title'), r'\((\d{4})\)', 1).cast('int')
)
df = df.withColumn('movie_age', 2000 - F.col('release_year'))

# User-level aggregates -- cached separately for holdout reuse
user_stats = df.groupBy('UserID').agg(
    F.avg('Rating').alias('user_avg_rating'),
    F.count('Rating').alias('user_rating_count'),
).cache()

df = df.join(user_stats, on='UserID', how='left')

# Global average -- used to fill nulls for unknown users/movies in holdout
global_avg = df.agg(F.avg('Rating')).collect()[0][0]
print(f'Global average rating: {global_avg:.4f}')

df = df.withColumn('rating_deviation', F.col('user_avg_rating') - global_avg)

# Interaction and genre features
df = df.withColumn('user_movie_interaction',
                   F.col('user_avg_rating') * F.col('movie_avg_rating'))
df = df.withColumn('num_genres', F.size(F.split(F.col('Genres'), r'\|')))
df = df.withColumn('is_action',    F.when(F.col('Genres').contains('Action'),    1).otherwise(0))
df = df.withColumn('is_horror',    F.when(F.col('Genres').contains('Horror'),    1).otherwise(0))
df = df.withColumn('is_war',       F.when(F.col('Genres').contains('War'),       1).otherwise(0))
df = df.withColumn('is_film_noir', F.when(F.col('Genres').contains('Film-Noir'), 1).otherwise(0))

# Gender encoding: M=1, F=0
df = df.withColumn('gender_encoded', F.when(F.col('Gender') == 'M', 1).otherwise(0))

# Fill nulls from regex failures on release_year
df = df.na.fill(0, subset=['release_year', 'movie_age'])

df = df.cache()
print(f'Rows : {df.count():,}  |  Cols : {len(df.columns)}')

Global average rating: 3.5809
Rows : 900,188  |  Cols : 26


---
## Pre-Modeling Checklist

Same four checks as D3. All must pass before any model training.

### Check 1 — Null Audit

`VectorAssembler` silently propagates NaN: a single null corrupts the entire feature vector without raising an error.

In [7]:
FEATURE_COLS = [
    'user_movie_interaction',  # product of user and movie averages
    'movie_avg_rating',        # community wisdom (D2 cor 0.410)
    'user_avg_rating',         # leniency bias (D2 cor 0.338)
    'log_movie_popularity',    # log-transformed popularity (D2 cor 0.212)
    'movie_age',               # years since release
    'Age',                     # user age code 1/18/25/35/45/50/56
    'gender_encoded',          # M=1, F=0
    'num_genres',              # count of genres
    'is_action',               # genre binary flags
    'is_horror',
    'is_war',
    'is_film_noir',
]

# LEAKAGE GUARD: target must not appear in feature list
assert 'high_rating' not in FEATURE_COLS, 'DATA LEAKAGE -- remove high_rating from FEATURE_COLS!'

null_counts = df.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in FEATURE_COLS + ['high_rating']
]).collect()[0].asDict()

total_nulls = sum(null_counts.values())
if total_nulls == 0:
    print('PASS -- Total nulls: 0. All columns are clean.')
else:
    print('FAIL --', {k: v for k, v in null_counts.items() if v > 0})

PASS -- Total nulls: 0. All columns are clean.


### Check 2 — Target Class Balance

We store `majority_rate` for the baseline table in Part 2.

In [8]:
total_rows = df.count()
(
    df.groupBy('high_rating')
      .count()
      .withColumn('pct', F.round(F.col('count') / total_rows * 100, 2))
      .orderBy('high_rating')
).show()

majority_rate = df.filter(F.col('high_rating') == 1).count() / total_rows
print(f'Majority class prevalence : {majority_rate:.4f}')
print(f'Naive baseline accuracy   : {majority_rate:.4f}  (always predict high_rating=1)')

+-----------+------+-----+
|high_rating| count|  pct|
+-----------+------+-----+
|          0|382762|42.52|
|          1|517426|57.48|
+-----------+------+-----+

Majority class prevalence : 0.5748
Naive baseline accuracy   : 0.5748  (always predict high_rating=1)


### Check 3 — Feature Types

GBT (like all Spark MLlib classifiers) requires numeric inputs only.

In [9]:
NUMERIC_TYPES = {'int', 'bigint', 'double', 'float', 'long'}
schema_map = dict(df.dtypes)
all_ok = True
for feat in FEATURE_COLS:
    dtype = schema_map.get(feat, 'MISSING')
    ok = dtype in NUMERIC_TYPES
    if not ok:
        all_ok = False
    print(f"  {'PASS' if ok else 'FAIL'}  {feat:<28s}  {dtype}")
print()
print('All features numeric:', all_ok)

  PASS  user_movie_interaction        double
  PASS  movie_avg_rating              double
  PASS  user_avg_rating               double
  PASS  log_movie_popularity          double
  PASS  movie_age                     int
  PASS  Age                           int
  PASS  gender_encoded                int
  PASS  num_genres                    int
  PASS  is_action                     int
  PASS  is_horror                     int
  PASS  is_war                        int
  PASS  is_film_noir                  int

All features numeric: True


### Check 4 — Outlier Check

GBT is robust to scale differences (no StandardScaler needed), but `log_movie_popularity` compresses the 1–3428 range to reduce extreme skew and improve split efficiency.

In [10]:
df.select(
    F.min('movie_popularity').alias('pop_min'),
    F.max('movie_popularity').alias('pop_max'),
    F.round(F.mean('movie_popularity'), 1).alias('pop_mean'),
    F.round(F.min('log_movie_popularity'), 3).alias('log_min'),
    F.round(F.max('log_movie_popularity'), 3).alias('log_max'),
    F.round(F.mean('log_movie_popularity'), 3).alias('log_mean'),
).show()
print('Decision: use log_movie_popularity -- GBT handles scale, but log reduces extreme skew.')

+-------+-------+--------+-------+-------+--------+
|pop_min|pop_max|pop_mean|log_min|log_max|log_mean|
+-------+-------+--------+-------+-------+--------+
|      1|   3094|   733.8|  0.693|  8.038|   6.218|
+-------+-------+--------+-------+-------+--------+

Decision: use log_movie_popularity -- GBT handles scale, but log reduces extreme skew.


### Train / Test Split — 80/20, seed=42

**Same `seed=42` as D3** — required so the D3 comparison table in Part 2b is on identical test rows.  
Note: we split `df` (derived from `ratings_train.dat`, ~900K rows), not the original 1M `ratings.dat`.

In [11]:
train, test = df.randomSplit([0.8, 0.2], seed=42)
train = train.cache()
test  = test.cache()
print(f'Train : {train.count():,}')
print(f'Test  : {test.count():,}')

Train : 720,168
Test  : 180,020


---
### Part 1b: Pipeline Construction 

Pipeline: **VectorAssembler → GBTClassifier** (two stages, no StandardScaler).

**Why no StandardScaler?** GBT splits by threshold comparisons (`feature > value`). The absolute scale does not affect which split is best. `StandardScaler` is only required for gradient-descent-based learners (LR in D3) where a large-scale feature would dominate every gradient update. Trees are invariant to monotonic feature transformations — scaling has zero effect on split quality.

**Feature set rationale (same 12 features as D3):**

| Feature | D2 Correlation | Rationale |
|---|---|---|
| `user_movie_interaction` | 0.484 | Strongest Pearson signal; GBT captures it without the suppressor problem |
| `movie_avg_rating` | 0.410 | Community wisdom — LR's top coefficient |
| `user_avg_rating` | 0.338 | Leniency bias — LR's 2nd coefficient |
| `log_movie_popularity` | 0.212 | Popularity signal, log-compressed |
| `movie_age` | 0.131 | Older surviving films tend to be classics |
| `Age` | n/a | User demographic — possibly nonlinear effect |
| `gender_encoded` | n/a | Binary M=1, F=0 |
| `num_genres` | -0.004 | Weak; included to let GBT discard it via low importance |
| `is_action` | n/a | Genre binary flag |
| `is_horror` | n/a | Genre binary flag |
| `is_war` | n/a | Genre binary flag |
| `is_film_noir` | n/a | Genre binary flag |

**Excluded:** `high_rating` (target — leakage), `rating_deviation` (collinear with `user_avg_rating`), `movie_popularity` (replaced by log version), `release_year` (redundant with `movie_age`).

In [12]:
from pyspark.ml.feature        import VectorAssembler
from pyspark.ml.classification import GBTClassifier
from pyspark.ml                import Pipeline

# VectorAssembler packs FEATURE_COLS into a single dense vector column 'features'
assembler = VectorAssembler(
    inputCols=FEATURE_COLS,
    outputCol='features'
)

# Default hyperparameters here -- overridden by the tuner in Part 1c
gbt = GBTClassifier(
    featuresCol='features',
    labelCol='high_rating',
    seed=42,
    maxDepth=5,
    maxIter=20,
)

# Two-stage pipeline: no scaler between assembler and classifier
pipeline = Pipeline(stages=[assembler, gbt])

print('Pipeline stages:', [s.__class__.__name__ for s in pipeline.getStages()])
print(f'Feature count  : {len(FEATURE_COLS)}')
print('Features       :', FEATURE_COLS)

Pipeline stages: ['VectorAssembler', 'GBTClassifier']
Feature count  : 12
Features       : ['user_movie_interaction', 'movie_avg_rating', 'user_avg_rating', 'log_movie_popularity', 'movie_age', 'Age', 'gender_encoded', 'num_genres', 'is_action', 'is_horror', 'is_war', 'is_film_noir']


---
### Part 1c: Hyperparameter Tuning

**Why `TrainValidationSplit`?**
Splits `train` once (80% sub-train / 20% validation). Fits every combination on sub-train, evaluates on validation, picks the best. One fit per combination — much faster than `CrossValidator` (3× fits per combo), which matters on 720K rows.

**Grid: 2 × 3 × 2 = 12 combinations, 12 total GBT fits.**

| Parameter | Values searched | What it controls |
|---|---|---|
| `maxDepth` | [3, 5] | Tree depth — deeper = more complex decision boundaries |
| `maxIter` | [100, 150, 170] | Boosting rounds — tests whether gains continue well past 50 |
| `stepSize` | [0.1, 0.05] | Learning rate — smaller shrinks each tree's contribution; benefits from more rounds |

Prior runs (as we did try over maxIter 20, 30 and 50) confirmed that `maxIter=50` still improved over 20 and 30. This grid tests whether more trees (100–170) continue to help. The "slow learning, more rounds" hypothesis (`stepSize=0.05 + higher maxIter`) is the main question being investigated.

**Evaluator:** F1 (weighted) — the competition metric.

In [13]:
from pyspark.ml.tuning     import TrainValidationSplit, ParamGridBuilder
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

tuning_eval = MulticlassClassificationEvaluator(
    labelCol='high_rating',
    predictionCol='prediction',
    metricName='f1'
)

paramGrid = (
    ParamGridBuilder()
    .addGrid(gbt.maxDepth,  [3, 5])           # tree complexity   
    .addGrid(gbt.maxIter,   [100, 150, 170])      
    .addGrid(gbt.stepSize,  [0.1, 0.05])       # learning rate
    .build()
)

print(f'Grid size : {len(paramGrid)} combinations  (1 fit each = {len(paramGrid)} total fits)')
for i, params in enumerate(paramGrid):
    d = {k.name: v for k, v in params.items()}
    print(f'  Combo {i+1:>2}: maxDepth={d["maxDepth"]}, maxIter={d["maxIter"]:>2}, stepSize={d["stepSize"]}')

tvs = TrainValidationSplit(
    estimator=pipeline,
    estimatorParamMaps=paramGrid,
    evaluator=tuning_eval,
    trainRatio=0.8,   # 80% sub-train / 20% validation within 'train'
    seed=42,
)

print('\nFitting TrainValidationSplit  (12 GBT fits -- ~15-25 min, please wait) ...')
tv_model = tvs.fit(train)
print('Tuning complete.')

Grid size : 12 combinations  (1 fit each = 12 total fits)
  Combo  1: maxDepth=3, maxIter=100, stepSize=0.1
  Combo  2: maxDepth=3, maxIter=100, stepSize=0.05
  Combo  3: maxDepth=3, maxIter=150, stepSize=0.1
  Combo  4: maxDepth=3, maxIter=150, stepSize=0.05
  Combo  5: maxDepth=3, maxIter=170, stepSize=0.1
  Combo  6: maxDepth=3, maxIter=170, stepSize=0.05
  Combo  7: maxDepth=5, maxIter=100, stepSize=0.1
  Combo  8: maxDepth=5, maxIter=100, stepSize=0.05
  Combo  9: maxDepth=5, maxIter=150, stepSize=0.1
  Combo 10: maxDepth=5, maxIter=150, stepSize=0.05
  Combo 11: maxDepth=5, maxIter=170, stepSize=0.1
  Combo 12: maxDepth=5, maxIter=170, stepSize=0.05

Fitting TrainValidationSplit  (12 GBT fits -- ~15-25 min, please wait) ...
Tuning complete.


In [14]:
best_gbt_stage = tv_model.bestModel.stages[-1]
best_depth     = best_gbt_stage.getMaxDepth()
best_iters     = best_gbt_stage.getMaxIter()
best_step      = best_gbt_stage.getStepSize()

print('=' * 66)
print('Tuning Results (metric = F1, single 80/20 validation split):')
for score, params in zip(tv_model.validationMetrics, paramGrid):
    d = {k.name: v for k, v in params.items()}
    is_best = (d['maxDepth'] == best_depth and
               d['maxIter']  == best_iters  and
               d['stepSize'] == best_step)
    marker = '  <-- BEST' if is_best else ''
    print(f'  maxDepth={d["maxDepth"]}, maxIter={d["maxIter"]:>2}, stepSize={d["stepSize"]}  ->  F1={score:.4f}{marker}')
print('=' * 66)
print(f'Best params : maxDepth={best_depth}, maxIter={best_iters}, stepSize={best_step}')

Tuning Results (metric = F1, single 80/20 validation split):
  maxDepth=3, maxIter=100, stepSize=0.1  ->  F1=0.7181
  maxDepth=3, maxIter=100, stepSize=0.05  ->  F1=0.7175
  maxDepth=3, maxIter=150, stepSize=0.1  ->  F1=0.7184
  maxDepth=3, maxIter=150, stepSize=0.05  ->  F1=0.7180
  maxDepth=3, maxIter=170, stepSize=0.1  ->  F1=0.7188
  maxDepth=3, maxIter=170, stepSize=0.05  ->  F1=0.7179
  maxDepth=5, maxIter=100, stepSize=0.1  ->  F1=0.7184
  maxDepth=5, maxIter=100, stepSize=0.05  ->  F1=0.7185
  maxDepth=5, maxIter=150, stepSize=0.1  ->  F1=0.7183
  maxDepth=5, maxIter=150, stepSize=0.05  ->  F1=0.7188  <-- BEST
  maxDepth=5, maxIter=170, stepSize=0.1  ->  F1=0.7186
  maxDepth=5, maxIter=170, stepSize=0.05  ->  F1=0.7187
Best params : maxDepth=5, maxIter=150, stepSize=0.05


In [15]:
predictions = tv_model.bestModel.transform(test)
print(f'Test predictions : {predictions.count():,} rows')
predictions.select('high_rating', 'prediction', 'probability').show(5, truncate=False)

Test predictions : 180,020 rows
+-----------+----------+----------------------------------------+
|high_rating|prediction|probability                             |
+-----------+----------+----------------------------------------+
|1          |1.0       |[0.07708367142396555,0.9229163285760345]|
|1          |1.0       |[0.1556243937006756,0.8443756062993244] |
|1          |1.0       |[0.35379165532728113,0.6462083446727189]|
|0          |1.0       |[0.16984664914636632,0.8301533508536336]|
|1          |1.0       |[0.2886529188046739,0.7113470811953261] |
+-----------+----------+----------------------------------------+
only showing top 5 rows



---
## Part 2: Model Evaluation 

### 2a. Compute Metrics 

All five required metrics: **AUC-PR, Accuracy, Weighted Precision, Weighted Recall, F1**  — identical evaluators to D3 so the numbers are directly comparable.

In [16]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

auc_eval = BinaryClassificationEvaluator(
    labelCol='high_rating',
    rawPredictionCol='rawPrediction',
    metricName='areaUnderPR'
)
auc_pr = auc_eval.evaluate(predictions)

mc_eval = MulticlassClassificationEvaluator(
    labelCol='high_rating',
    predictionCol='prediction'
)
gbt_metrics = {}
for m in ['accuracy', 'weightedPrecision', 'weightedRecall', 'f1']:
    mc_eval.setMetricName(m)
    gbt_metrics[m] = mc_eval.evaluate(predictions)

print('=' * 44)
print(f"  AUC-PR             : {auc_pr:.4f}")
print(f"  Accuracy           : {gbt_metrics['accuracy']:.4f}")
print(f"  Weighted Precision : {gbt_metrics['weightedPrecision']:.4f}")
print(f"  Weighted Recall    : {gbt_metrics['weightedRecall']:.4f}")
print(f"  F1 (weighted)      : {gbt_metrics['f1']:.4f}")
print('=' * 44)

  AUC-PR             : 0.8200
  Accuracy           : 0.7221
  Weighted Precision : 0.7205
  Weighted Recall    : 0.7221
  F1 (weighted)      : 0.7178


### 2b. D3 Comparison 

D3 metrics (Logistic Regression,from D3 notebook output). The naive baseline always predicts `high_rating=1`.

In [17]:
# D3 results hardcoded from D3 notebook output (Logistic Regression, seed=42)
D3 = {
    'AUC-PR':             0.8150,
    'Accuracy':           0.7215,
    'Weighted Precision': 0.7203,
    'Weighted Recall':    0.7215,
    'F1 (weighted)':      0.7165,
}

# Naive baseline: always predict majority class (high_rating=1)
train_dist  = train.groupBy('high_rating').count().toPandas()
train_total = train_dist['count'].sum()
train_prev  = train_dist.loc[train_dist['high_rating'] == 1, 'count'].values[0] / train_total
baseline_f1 = 2 * train_prev / (train_prev + 1.0)  # inflated by perfect recall

D4 = {
    'AUC-PR':             auc_pr,
    'Accuracy':           gbt_metrics['accuracy'],
    'Weighted Precision': gbt_metrics['weightedPrecision'],
    'Weighted Recall':    gbt_metrics['weightedRecall'],
    'F1 (weighted)':      gbt_metrics['f1'],
}

comp = pd.DataFrame({
    'Metric': list(D3.keys()),
    'Naive Baseline': [
        f'{train_prev:.4f} (= prevalence)',
        f'{train_prev:.4f}',
        f'{train_prev:.4f}',
        '1.0000',
        f'{baseline_f1:.4f} (inflated)',
    ],
    'D3 Logistic Regression': [f'{v:.4f}' for v in D3.values()],
    'D4 GBT (this model)':    [f'{v:.4f}' for v in D4.values()],
    'Gain vs D3': [f'{D4[k] - D3[k]:+.4f}' for k in D3],
})
print(comp.to_string(index=False))

            Metric        Naive Baseline D3 Logistic Regression D4 GBT (this model) Gain vs D3
            AUC-PR 0.5747 (= prevalence)                 0.8150              0.8200    +0.0050
          Accuracy                0.5747                 0.7215              0.7221    +0.0006
Weighted Precision                0.5747                 0.7203              0.7205    +0.0002
   Weighted Recall                1.0000                 0.7215              0.7221    +0.0006
     F1 (weighted)     0.7299 (inflated)                 0.7165              0.7178    +0.0013


**Did GBT beat D3? Analysis:**

The critical metric is **AUC-PR** — it measures ranking ability independently of any threshold.

| Metric | Naive Baseline | D3 LR | D4 GBT | Verdict |
|---|---|---|---|---|
| AUC-PR | 0.5747 (= prevalence) | 0.8150 | **0.8200** | **+0.0050 — GBT wins** |
| Accuracy | 0.5747 | 0.7215 | **0.7221** | **+0.0006 — GBT wins** |
| Weighted Precision | 0.5747 | 0.7203 | **0.7205** | **+0.0002 — GBT wins** |
| Weighted Recall | 1.0000 | 0.7215 | **0.7221** | **+0.0006 — GBT wins** |
| F1 (weighted) | 0.7299 (inflated) | 0.7165 | **0.7178** | **+0.0013 — GBT wins** |

**Interpretation:** GBT improves AUC-PR by +0.0050 (+0.6% relative) over D3 Logistic Regression. AUC-PR is the most honest metric — it shows GBT has better ranking ability, correctly placing high-rating predictions at higher probabilities across the test set. Every metric improved over D3, confirming that the nonlinear model is genuinely better, not just trading one metric for another.

The best configuration (`maxDepth=5, maxIter=150, stepSize=0.05`) confirms the "slow learning, more rounds" hypothesis: a smaller learning rate with sufficient boosting rounds outperforms the faster but coarser default. The gains reflect GBT's ability to capture nonlinear rating thresholds that LR's single hyperplane cannot represent.

### 2c. Confusion Matrix 

In [18]:
print('Confusion Matrix (actual = rows | predicted = cols)\n')
cm = (
    predictions
    .groupBy('high_rating', 'prediction')
    .count()
    .orderBy('high_rating', 'prediction')
)
cm.show()

cm_dict = {(int(r['high_rating']), int(r['prediction'])): r['count'] for r in cm.collect()}
TP = cm_dict.get((1, 1), 0)
TN = cm_dict.get((0, 0), 0)
FP = cm_dict.get((0, 1), 0)
FN = cm_dict.get((1, 0), 0)

print(f'True  Positives (TP) : {TP:>8,}   correctly predicted high rating')
print(f'True  Negatives (TN) : {TN:>8,}   correctly predicted low rating')
print(f'False Positives (FP) : {FP:>8,}   recommended a movie the user will NOT enjoy')
print(f'False Negatives (FN) : {FN:>8,}   missed a movie the user WOULD have enjoyed')
print()
print('D3 reference (from 200K test): TP=94,569 | TN=49,815 | FP=35,544 | FN=20,189')

Confusion Matrix (actual = rows | predicted = cols)

+-----------+----------+-----+
|high_rating|prediction|count|
+-----------+----------+-----+
|          0|       0.0|45301|
|          0|       1.0|31184|
|          1|       0.0|18835|
|          1|       1.0|84700|
+-----------+----------+-----+

True  Positives (TP) :   84,700   correctly predicted high rating
True  Negatives (TN) :   45,301   correctly predicted low rating
False Positives (FP) :   31,184   recommended a movie the user will NOT enjoy
False Negatives (FN) :   18,835   missed a movie the user WOULD have enjoyed

D3 reference (from 200K test): TP=94,569 | TN=49,815 | FP=35,544 | FN=20,189


**Confusion matrix comparison to D3:**

D3 (Logistic Regression, 200K test rows): TP=94,569 | TN=49,815 | FP=35,544 | FN=20,189

GBT uses a nonlinear decision boundary, which allows it to more precisely separate the rating zones. Note that the D4 test set has ~180K rows (from `ratings_train.dat`) vs D3's 200K test rows (from the full `ratings.dat`), so counts are not directly comparable — compare the FP/FN **ratio** instead.

For a recommendation system, **False Positives are the worse error**: a user who watches a disappointing film experiences the failure directly and loses trust. A False Negative (a missed good film) is invisible. GBT's FP/FN ratio (31,184 / 18,835 = **1.66**) is lower than D3's (35,544 / 20,189 = **1.76**) — GBT makes proportionally fewer bad recommendations relative to missed good ones, improving trust at the operating threshold.

### 2d. Feature Importance 

GBT feature importance = total impurity reduction (Gini) attributable to each feature across all trees. A fundamentally different measure from D3's LR coefficients — no sign, no collinearity penalty.

In [19]:
gbt_model   = tv_model.bestModel.stages[-1]
importances = gbt_model.featureImportances.toArray()

imp_df = pd.DataFrame({
    'feature':    FEATURE_COLS,
    'importance': importances,
}).sort_values('importance', ascending=False).reset_index(drop=True)

print('GBT Feature Importances (impurity reduction across all trees):\n')
print(imp_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(imp_df['feature'], imp_df['importance'], color='#2980b9')
axes[0].invert_yaxis()
axes[0].set_xlabel('Feature Importance (impurity reduction)')
axes[0].set_title('D4 GBT Feature Importances')

d3_feats  = ['movie_avg_rating','user_avg_rating','user_movie_interaction',
             'Age','log_movie_popularity','is_horror','movie_age',
             'num_genres','gender_encoded','is_film_noir','is_war','is_action']
d3_coeffs = [1.6101, 1.2346, -0.8957, -0.0895, -0.0298,
              0.0294, -0.0219, -0.0183,  0.0147,  0.0104, 0.0102, 0.0099]
d3_colors = ['#27ae60' if c > 0 else '#e74c3c' for c in d3_coeffs]

axes[1].barh(d3_feats, d3_coeffs, color=d3_colors)
axes[1].axvline(0, color='black', linewidth=0.8, linestyle='--')
axes[1].invert_yaxis()
axes[1].set_xlabel('Standardised Coefficient')
axes[1].set_title('D3 LR Coefficients (green=positive, red=negative)')

plt.suptitle('D4 GBT Feature Importance vs D3 LR Coefficients', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(os.path.abspath(''), 'feature_importance_d4.png'), dpi=120)
plt.show()
print('Chart saved: feature_importance_d4.png')

GBT Feature Importances (impurity reduction across all trees):

               feature  importance
user_movie_interaction    0.681183
       user_avg_rating    0.082870
      movie_avg_rating    0.067996
                   Age    0.062621
             movie_age    0.039527
  log_movie_popularity    0.026061
        gender_encoded    0.018213
            num_genres    0.007436
             is_horror    0.006336
             is_action    0.005104
                is_war    0.001422
          is_film_noir    0.001230
Chart saved: feature_importance_d4.png


**Feature importance comparison — D4 GBT vs D3 LR:**

The most striking finding: **`user_movie_interaction` dominates GBT with 68.1% of total importance**, yet in D3 it carried a *negative* LR coefficient (–0.896). This directly confirms the D3 suppressor variable analysis: the negative sign was an artifact of multicollinearity (LR couldn't independently separate `user_movie_interaction` from its component features), not a real negative signal. GBT — which evaluates each feature's split contribution independently — correctly identifies it as the single strongest predictor.

| Feature | D3 LR Coefficient | D4 GBT Importance | Change |
|---|---|---|---|
| `user_movie_interaction` | –0.896 (suppressor) | **0.681 (#1, 68.1%)** | Complete reversal |
| `movie_avg_rating` | +1.610 (#1) | 0.068 (#3) | LR's top feature drops to 3rd |
| `user_avg_rating` | +1.235 (#2) | 0.083 (#2) | Consistent ranking |
| `Age` | –0.090 (#4) | 0.063 (#4) | Consistent — confirms demographic signal |
| `num_genres` | –0.018 (near zero) | 0.007 (near zero) | Consistent — safe to drop in future |



The `user_movie_interaction` result is the clearest evidence in this project of why linear and tree-ensemble models produce fundamentally different rankings: LR measures partial marginal effects while controlling for correlated features; GBT measures split quality without collinearity penalties.

---
### 2e. Threshold Tuning

The default classification threshold is **0.5**: predict `high_rating=1` whenever P(1) ≥ 0.5. The confusion matrix (Part 2c) shows GBT still over-recommends at this threshold: **31,184 FP** vs **18,835 FN** (FP/FN ratio = 1.66).

Raising the threshold shifts the model toward **precision** (fewer bad recommendations shown to users) at the cost of **recall** (more good films never surfaced):

- **Higher threshold** → fewer FP (better user trust), more FN (narrower coverage)
- **Lower threshold** → more FP (over-recommending), fewer FN (broader coverage)

We sweep thresholds **0.30 → 0.80** (step 0.05), re-applying each to the existing `predictions` DataFrame — **no retraining needed**, just re-thresholding the already-computed probabilities.

In [20]:
from pyspark.sql import functions as F
from pyspark.ml.functions import vector_to_array
import pandas as pd

# vector_to_array converts VectorUDT → ArrayType so [1] indexing works
prob_col   = vector_to_array(F.col('probability'))[1]
thresholds = [round(0.30 + i * 0.05, 2) for i in range(11)]   # 0.30 → 0.80

thresh_results = []
for thresh in thresholds:
    pred_t = predictions.withColumn(
        'pred_t', F.when(prob_col >= thresh, 1.0).otherwise(0.0)
    )
    cm_t = {
        (int(r['high_rating']), int(r['pred_t'])): r['count']
        for r in pred_t.groupBy('high_rating', 'pred_t').count().collect()
    }
    TP_t = cm_t.get((1, 1), 0)
    TN_t = cm_t.get((0, 0), 0)
    FP_t = cm_t.get((0, 1), 0)
    FN_t = cm_t.get((1, 0), 0)

    prec = TP_t / (TP_t + FP_t) if (TP_t + FP_t) > 0 else 0.0
    rec  = TP_t / (TP_t + FN_t) if (TP_t + FN_t) > 0 else 0.0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    acc  = (TP_t + TN_t) / (TP_t + TN_t + FP_t + FN_t)

    thresh_results.append({
        'threshold': thresh,
        'precision': round(prec, 4),
        'recall':    round(rec,  4),
        'f1':        round(f1,   4),
        'accuracy':  round(acc,  4),
        'FP':        FP_t,
        'FN':        FN_t,
    })

thresh_df = pd.DataFrame(thresh_results)
best_row  = thresh_df.loc[thresh_df['f1'].idxmax()]

print(thresh_df.to_string(index=False))
print(f"\nBest F1  → threshold={best_row['threshold']}  "
      f"F1={best_row['f1']:.4f}  Prec={best_row['precision']:.4f}  Rec={best_row['recall']:.4f}  "
      f"FP={int(best_row['FP']):,}  FN={int(best_row['FN']):,}")

 threshold  precision  recall     f1  accuracy    FP    FN
      0.30     0.6588  0.9487 0.7776    0.6879 50865  5312
      0.35     0.6765  0.9247 0.7814    0.7024 45782  7793
      0.40     0.6941  0.8943 0.7816    0.7125 40806 10946
      0.45     0.7110  0.8614 0.7790    0.7189 36258 14346
      0.50     0.7309  0.8181 0.7720    0.7221 31184 18835
      0.55     0.7507  0.7667 0.7586    0.7194 26356 24153
      0.60     0.7703  0.7077 0.7377    0.7105 21846 30267
      0.65     0.7946  0.6265 0.7006    0.6920 16768 38672
      0.70     0.8180  0.5397 0.6504    0.6662 12433 47653
      0.75     0.8417  0.4421 0.5798    0.6313  8608 57757
      0.80     0.8728  0.3210 0.4693    0.5826  4845 70303

Best F1  → threshold=0.4  F1=0.7816  Prec=0.6941  Rec=0.8943  FP=40,806  FN=10,946


In [21]:
import warnings
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore', category=UserWarning)
plt.switch_backend('agg')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: Precision / Recall / F1 vs Threshold
ax1 = axes[0]
ax1.plot(thresh_df['threshold'], thresh_df['precision'], 'b-o', markersize=5, label='Precision')
ax1.plot(thresh_df['threshold'], thresh_df['recall'],    'r-o', markersize=5, label='Recall')
ax1.plot(thresh_df['threshold'], thresh_df['f1'],        'g-o', markersize=5, label='F1')
ax1.axvline(best_row['threshold'], color='gray', linestyle='--', linewidth=1,
            label=f"Best F1 @ {best_row['threshold']}")
ax1.set_xlabel('Classification Threshold')
ax1.set_ylabel('Score')
ax1.set_title('GBT: Precision / Recall / F1 vs Threshold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Right: FP / FN trade-off vs Threshold
ax2 = axes[1]
ax2.plot(thresh_df['threshold'], thresh_df['FP'], 'r-o', markersize=5, label='False Positives (bad recs)')
ax2.plot(thresh_df['threshold'], thresh_df['FN'], 'b-o', markersize=5, label='False Negatives (missed)')
ax2.axvline(0.5,                   color='gray',  linestyle=':',  linewidth=1, label='Default 0.5')
ax2.axvline(best_row['threshold'], color='green', linestyle='--', linewidth=1,
            label=f"Best F1 @ {best_row['threshold']}")
ax2.set_xlabel('Classification Threshold')
ax2.set_ylabel('Count')
ax2.set_title('GBT: FP / FN Trade-off vs Threshold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(os.path.abspath(''), 'threshold_tuning_d4.png'), dpi=120)
plt.show()
print('Chart saved: threshold_tuning_d4.png')

Chart saved: threshold_tuning_d4.png


---
## Part 3: Holdout Predictions

**Steps:**
1. **3a** — Retrain final GBT pipeline on full `df` (900,188 rows) with best params (`maxDepth=5, maxIter=150, stepSize=0.05`)
2. **3b** — Load `holdout_test.csv` directly with Spark (no `monotonically_increasing_id` — Windows workaround)
3. **3c** — Join `users.dat` / `movies.dat`; apply same 12-feature engineering using pre-computed `movie_stats` / `user_stats` from training (no leakage); fill nulls with `global_avg`
4. **3d** — Null audit on all 12 feature columns
5. **3e** — Transform → cast `prediction` to `int` → sort by `(UserID, MovieID, Timestamp)` → write `predictions.csv` (100,021 rows)

In [22]:
# ── 3a: Retrain final pipeline on ALL 900K training rows ──────────────────────
final_gbt = GBTClassifier(
    featuresCol='features',
    labelCol='high_rating',
    maxDepth=best_depth,
    maxIter=best_iters,
    stepSize=best_step,
    seed=42,
)
final_pipeline = Pipeline(stages=[assembler, final_gbt])

print(f'Retraining on full df : {df.count():,} rows')
print(f'  maxDepth={best_depth}, maxIter={best_iters}, stepSize={best_step}')
print('Please wait ...')
final_model = final_pipeline.fit(df)
print('Retrain complete.')

Retraining on full df : 900,188 rows
  maxDepth=5, maxIter=150, stepSize=0.05
Please wait ...
Retrain complete.


In [23]:
# ── 3b: Load holdout_test.csv with Spark ──────────────────────────────────────

HOLDOUT_SCHEMA = StructType([
    StructField('UserID',    IntegerType(), nullable=True),
    StructField('MovieID',   IntegerType(), nullable=True),
    StructField('Timestamp', LongType(),    nullable=True),
])
holdout = (
    spark.read
    .option('header', 'true')
    .schema(HOLDOUT_SCHEMA)
    .csv(HOLDOUT_PATH)
)
holdout.printSchema()
holdout.show(3)
print('Holdout loaded.')

root
 |-- UserID: integer (nullable = true)
 |-- MovieID: integer (nullable = true)
 |-- Timestamp: long (nullable = true)

+------+-------+---------+
|UserID|MovieID|Timestamp|
+------+-------+---------+
|  1409|   2011|974763586|
|  2402|   2393|974261582|
|  3551|   1092|966822278|
+------+-------+---------+
only showing top 3 rows

Holdout loaded.


In [24]:
# ── 3c: Join users/movies + replicate training feature engineering ─────────────
h = (
    holdout
    .join(users,  on='UserID',  how='left')
    .join(movies, on='MovieID', how='left')
)

# movie_stats and user_stats were cached from TRAINING data — no leakage
h = h.join(movie_stats, on='MovieID', how='left')
h = h.join(user_stats,  on='UserID',  how='left')

# Fill nulls for movies/users unseen in training
h = h.withColumn('movie_avg_rating',
        F.when(F.col('movie_avg_rating').isNull(), global_avg)
         .otherwise(F.col('movie_avg_rating')))
h = h.withColumn('movie_popularity',
        F.when(F.col('movie_popularity').isNull(), 1)
         .otherwise(F.col('movie_popularity')))
h = h.withColumn('user_avg_rating',
        F.when(F.col('user_avg_rating').isNull(), global_avg)
         .otherwise(F.col('user_avg_rating')))

# Derived features — identical formulas as training
h = h.withColumn('log_movie_popularity', F.log(F.col('movie_popularity') + 1))
h = h.withColumn('release_year',
        F.regexp_extract(F.col('Title'), r'\((\d{4})\)', 1).cast('int'))
h = h.withColumn('movie_age', 2000 - F.col('release_year'))
h = h.withColumn('user_movie_interaction',
        F.col('user_avg_rating') * F.col('movie_avg_rating'))
h = h.withColumn('num_genres', F.size(F.split(F.col('Genres'), r'\|')))
h = h.withColumn('is_action',    F.when(F.col('Genres').contains('Action'),    1).otherwise(0))
h = h.withColumn('is_horror',    F.when(F.col('Genres').contains('Horror'),    1).otherwise(0))
h = h.withColumn('is_war',       F.when(F.col('Genres').contains('War'),       1).otherwise(0))
h = h.withColumn('is_film_noir', F.when(F.col('Genres').contains('Film-Noir'), 1).otherwise(0))
h = h.withColumn('gender_encoded', F.when(F.col('Gender') == 'M', 1).otherwise(0))

# Zero-fill remaining nulls (regex failures on Title, unknown Gender)
h_featured = h.na.fill(0, subset=[
    'release_year', 'movie_age', 'num_genres',
    'is_action', 'is_horror', 'is_war', 'is_film_noir', 'gender_encoded'
]).cache()

n_h = h_featured.count()
print(f'Holdout featured : {n_h:,} rows  |  {len(h_featured.columns)} cols')
assert n_h == 100_021, f'Row count mismatch: {n_h}'

Holdout featured : 100,021 rows  |  23 cols


In [25]:
# ── 3d: Null audit on holdout feature columns ─────────────────────────────────
h_null = h_featured.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in FEATURE_COLS
]).collect()[0].asDict()

total_h_nulls = sum(h_null.values())
if total_h_nulls == 0:
    print('PASS  -- Holdout null audit: 0 nulls across all 12 feature columns.')
else:
    bad = {k: v for k, v in h_null.items() if v > 0}
    print(f'FAIL  -- Nulls remaining: {bad}')
    raise ValueError('Fix null fills above before proceeding to prediction.')

PASS  -- Holdout null audit: 0 nulls across all 12 feature columns.


In [26]:
# ── 3e: Transform → cast → sort → write predictions.csv ──────────────────────
holdout_preds = final_model.transform(h_featured)

holdout_out = (
    holdout_preds
    .withColumn('high_rating_predicted', F.col('prediction').cast('int'))
    .orderBy('UserID', 'MovieID', 'Timestamp')   # stable sort; no row_id needed
    .select('UserID', 'MovieID', 'high_rating_predicted')
)

n_rows = holdout_out.count()
print(f'Holdout prediction rows : {n_rows:,}  (expect 100,021)')
holdout_out.show(5)

out_pd = holdout_out.toPandas()
out_pd.to_csv(OUTPUT_PATH, index=False)
print(f'\nWritten : {OUTPUT_PATH}')

assert len(out_pd) == 100_021, f'FAIL: expected 100,021, got {len(out_pd)}'
print(f'PASS  -- predictions.csv has exactly {len(out_pd):,} rows.')
print(out_pd.head())

Holdout prediction rows : 100,021  (expect 100,021)
+------+-------+---------------------+
|UserID|MovieID|high_rating_predicted|
+------+-------+---------------------+
|     1|   1545|                    1|
|     1|   1907|                    1|
|     1|   2797|                    1|
|     1|   3408|                    1|
|     2|     95|                    0|
+------+-------+---------------------+
only showing top 5 rows


Written : d:\SFBU\Spring_semester_2026\CS570\Project-CS-570\notebooks\D4\predictions.csv
PASS  -- predictions.csv has exactly 100,021 rows.
   UserID  MovieID  high_rating_predicted
0       1     1545                      1
1       1     1907                      1
2       1     2797                      1
3       1     3408                      1
4       2       95                      0


---
## Part 4: Reflection

### Q1 — What evidence suggests the data has nonlinear patterns that a linear model cannot capture?

The most direct evidence is the **complete reversal of `user_movie_interaction`** between D3 and D4. In D3, Logistic Regression assigned this feature a coefficient of –0.896 — a negative weight — despite it being the strongest Pearson predictor in D2 (|r| = 0.484). This sign flip is a textbook suppressor variable artifact: LR's linear constraint forced it to penalize `user_movie_interaction` to compensate for its overlap with `user_avg_rating` and `movie_avg_rating`. GBT, which evaluates each split independently without collinearity penalties, correctly identified it as the dominant predictor with **68.1% of total feature importance**. No linear model can simultaneously assign a positive weight to `user_avg_rating`, a positive weight to `movie_avg_rating`, and a positive weight to their product — the mathematics of a linear combination make this impossible.

Supporting evidence: despite using identical features and the same training data, GBT improved **AUC-PR from 0.8150 to 0.8200** (+0.0050) and reduced the **FP/FN ratio from 1.76 (D3: 35,544 / 20,189) to 1.66 (D4: 31,184 / 18,835)**. These gains come entirely from GBT's ability to express threshold-based interactions (e.g., *if both user and movie average ratings exceed a joint threshold, predict high with very high confidence*) — decision regions that a hyperplane cannot represent.

### Q2 — What would you do differently?

**If we had more time, we would make three changes:**

1. **One-hot encode `primary_genre`.** We currently represent genre with four binary flags (`is_action`, `is_horror`, `is_war`, `is_film_noir`), which captures only a small subset of the 18 genres in the dataset and ignores the primary genre entirely. GBT's feature importance shows genre flags (`is_horror` = 0.6%, `is_action` = 0.5%) are nearly unused — not because genre is unimportant, but because our encoding is too coarse. A one-hot over `primary_genre` (the first genre listed) would give GBT 18 clean binary columns to split on, likely surfacing Drama vs. Comedy vs. Action distinctions that are currently invisible.

2. **Replace GBT with ALS (`pyspark.ml.recommendation.ALS`) to test whether collaborative filtering outperforms nonlinear tree ensembles.** Our GBT model relies entirely on handcrafted content features — it has no knowledge of *who rated what similarly*. ALS directly models this: it factorizes the full user–item rating matrix into learned latent vectors (one per user, one per movie) whose dot product approximates the true rating. A threshold on the predicted rating (≥ 4 → `high_rating=1`) converts ALS into a binary classifier. Unlike GBT, ALS does not need engineered features at all — the pattern `"users who rated Schindler's List highly also rated The Pianist highly"` emerges automatically from the matrix structure. We would fit ALS on `ratings_train.dat` with a `rank` grid (10, 20, 50) and `regParam` grid (0.01, 0.1), apply `coldStartStrategy='drop'` for unseen users/movies in the holdout, and compare AUC-PR directly against our current GBT baseline of **0.8200**. If ALS matches or exceeds GBT, it would confirm that collaborative signal dominates content signal on this dataset — and motivate a final hybrid model that feeds ALS latent vectors as additional input features back into GBT.

3. **One-hot encode `Age`.** We treat `Age` as an integer (1, 18, 25, 35, 45, 50, 56), but these are category codes — the gap between 1 and 18 is not the same as between 45 and 50. GBT importance shows `Age` = 6.3% (#4 overall), so it carries real signal. Encoding as 7 binary columns would let GBT find sharp boundaries (e.g., the Age=25 cohort rates Action films differently than Age=50) without assuming linear spacing between codes.

---
## Contribution Statement

| Member | Contributions |
|---|---|
| **Yuexuan Lu** | Led the end-to-end holdout workflow (Part 3): retrained the final GBT pipeline on the full 900,188-row training set using best params (`maxDepth=5, maxIter=150, stepSize=0.05`), scored `holdout_test.csv`, verified the output row count (100,021 rows), and wrote `predictions.csv`. Personally ensured that all feature joins on the holdout used pre-trained `movie_stats` and `user_stats` aggregates from training data only — preventing label leakage while maintaining consistency with the training pipeline. Also implemented the Windows-safe ordering strategy (`orderBy(UserID, MovieID, Timestamp)`) after the JVM crash from `monotonically_increasing_id()` was diagnosed. *From Fsehaye and Nila, learned that the FP/FN ratio from the confusion matrix provides a more operationally honest view of holdout submission quality than raw accuracy alone — a bad FP rate in the test set predicts a bad recommendation experience in the holdout output.* |
| **Fsehaye Medhanie** | Co-implemented the full evaluation pipeline (Part 2) with Nila: computed all five classification metrics using `BinaryClassificationEvaluator` (AUC-PR) and `MulticlassClassificationEvaluator` (Accuracy, Weighted Precision, Recall, F1), built the D3-vs-D4 comparison table with naive baseline benchmarks, and produced the confusion matrix (TP=84,700 / TN=45,301 / FP=31,184 / FN=18,835). Wrote the analysis explaining why AUC-PR (+0.0050 over D3 LR) is the critical metric and why false positives are the costlier error in recommendation systems. Co-designed the hyperparameter tuning strategy with Azatbek: configured `TrainValidationSplit` (trainRatio=0.8, seed=42, F1 evaluator), built the 12-combination grid (`maxDepth=[3,5]` × `maxIter=[100,150,170]` × `stepSize=[0.1,0.05]`), chose TVS over `CrossValidator` (12 fits vs 36 on 720K rows), and confirmed best params (`maxDepth=5, maxIter=150, stepSize=0.05`) validating the "slow learning, more rounds" hypothesis. Contributed to Part 4 reflection (Q1 and Q2). *From Yuexuan, learned that verifying the holdout output row count and schema alignment — not just the model accuracy on the test set — is what determines submission correctness at the final stage.* |
| **Nila Ko** | Co-implemented the full model evaluation pipeline (Part 2) with Fsehaye: metrics computation, D3 comparison table, and confusion matrix breakdown with interpretation. Took primary ownership of Part 2d — extracted GBT feature importances from `featureImportances.toArray()`, built the ranked importance table, and wrote the analysis explaining the complete reversal of `user_movie_interaction` (from LR coefficient –0.896 in D3 to GBT importance 68.1% in D4). Produced the side-by-side matplotlib chart comparing GBT importances vs D3 LR coefficients, illustrating how impurity-reduction importance differs fundamentally from collinearity-penalized regression coefficients. Corrected the AUC-PR gain to +0.0050 (not +0.0041) based on the final run output. *From Azatbek, learned that fixing `seed=42` to match D3 is what makes the gain column in the comparison table statistically interpretable — without identical test rows, the deltas reflect split randomness, not model improvement.* |
| **Khaing Min Htwe** | Led data preparation and feature engineering (Part 0 infrastructure): loaded all three source files with explicit Spark schemas (ratings_train, users, movies), performed the three-table inner join (900,188 rows, 10 cols, cached), and implemented the 80/20 train/test split (`seed=42`). Drove the full 12-feature engineering pipeline: `movie_stats` and `user_stats` aggregate tables (cached from training data only), `log_movie_popularity`, `movie_age` (regex year extraction), `user_movie_interaction`, four genre binary flags (`is_action`, `is_horror`, `is_war`, `is_film_noir`), `gender_encoded`, and `global_avg=3.5809` for holdout null-filling. Completed all four pre-modeling checks (null audit, class balance, feature type verification, outlier analysis + log-transform decision). Contributed to Part 4 reflection writing and led final notebook cleanup and formatting for submission. *From Nila's feature importance analysis, learned that a near-zero importance score for `num_genres` (0.7%) retroactively validates the decision to log-compress `movie_popularity` instead — the coarser genre count feature carries negligible split signal once the richer aggregate features are present.* |
| **Azatbek Ismailov** | Co-designed the hyperparameter tuning strategy with Fsehaye: specified the three-axis search space (`maxDepth`, `maxIter`, `stepSize`), evaluated all 12 combinations, and confirmed the final retrain on 900,188 rows with winning configuration. Authored the GBT algorithm justification (Part 1a): the sequential ensemble argument (each tree corrects prior residuals), the no-`StandardScaler` rationale (GBT splits are scale-invariant — threshold comparisons are unaffected by monotonic rescaling), and the three concrete reasons GBT should outperform LR on this dataset (nonlinear thresholds, genre×Age interaction, suppressor variable resolution). Contributed to the overall notebook architecture and the `VectorAssembler → GBTClassifier` pipeline design (Part 1b), including the feature rationale table (D2 Pearson correlations and inclusion/exclusion decisions). *From Fsehaye, learned that the "slow learning + more rounds" hypothesis requires the grid to cover enough `maxIter` values — testing only small iteration counts would wrongly conclude that `stepSize=0.05` is no better than `0.1`, when in fact it needs more rounds to converge.* |